# Question 10: Emerging Algorithms & Complexity Problems

## Introduction

Cybersecurity and optimization systems regularly face problems that are computationally **hard** in the worst case. This notebook solves the 0/1 Knapsack problem exactly via dynamic programming, simulates the Traveling Salesman Problem (TSP) via brute force, and analyzes when exact methods become impractical.

## (a) Knapsack Problem (0/1, exact, via Dynamic Programming)

Given items with weights and values and a capacity `W`, find the maximum total value without exceeding the capacity, where each item may be taken at most once.

In [1]:
def knapsack(W, weights, values, n):
    dp = [[0] * (W + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        for w in range(W + 1):
            if i == 0 or w == 0:
                dp[i][w] = 0
            elif weights[i - 1] <= w:
                dp[i][w] = max(values[i - 1] + dp[i - 1][w - weights[i - 1]], dp[i - 1][w])
            else:
                dp[i][w] = dp[i - 1][w]
    return dp[n][W]

weights = [2, 3, 4, 5]
values  = [3, 4, 5, 6]
W = 5
result = knapsack(W, weights, values, len(weights))
print(f"Maximum value achievable with capacity {W}: {result}")

Maximum value achievable with capacity 5: 7


In [2]:
# ---- Verify the DP result against an exact brute-force search over all subsets ----
def knapsack_brute_force(W, weights, values, n):
    best = 0
    for mask in range(1 << n):
        w = sum(weights[i] for i in range(n) if mask & (1 << i))
        v = sum(values[i] for i in range(n) if mask & (1 << i))
        if w <= W:
            best = max(best, v)
    return best

bf_result = knapsack_brute_force(W, weights, values, len(weights))
print("Brute-force result:", bf_result)
assert result == bf_result

# 100 randomized trials, cross-validated
import random
random.seed(7)
for _ in range(100):
    n = random.randint(1, 12)
    cap = random.randint(1, 30)
    w = [random.randint(1, 15) for _ in range(n)]
    v = [random.randint(1, 20) for _ in range(n)]
    assert knapsack(cap, w, v, n) == knapsack_brute_force(cap, w, v, n)

print("DP result matches brute force on 100/100 randomized trials ✔")

Brute-force result: 7
DP result matches brute force on 100/100 randomized trials ✔


## (b) Traveling Salesman Problem (TSP) — Brute-Force Simulation

The TSP asks for the shortest possible round-trip route that visits every city exactly once and returns to the start. Brute force checks every permutation of cities — guaranteed optimal, but factorial in the number of cities.

In [3]:
from itertools import permutations

distance_matrix = [
    [0, 60, 135, 95],
    [60, 0, 190, 35],
    [135, 190, 0, 220],
    [95, 35, 220, 0],
]
cities = [0, 1, 2, 3]

best_cost = float('inf')
best_path = None
for path in permutations(cities[1:]):       # fix city 0 as the start to avoid counting rotations
    full_route = (0,) + path
    cost = sum(distance_matrix[full_route[i]][full_route[i+1]] for i in range(len(full_route) - 1))
    cost += distance_matrix[full_route[-1]][full_route[0]]   # return to start
    if cost < best_cost:
        best_cost, best_path = cost, full_route

print("Optimal TSP round-trip cost:", best_cost)
print("Optimal route (city indices):", best_path)

Optimal TSP round-trip cost:

 450
Optimal route (city indices): (0, 1, 3, 2)


In [4]:
# Cross-check: this is the SAME Rwanda city/distance setup used in Question 5's
# Dynamic Programming logistics solution. Both independently-coded algorithms should agree.
assert best_cost == 450, f"expected 450 km to match Q5's DP result, got {best_cost}"
print("VERIFIED: TSP brute-force result (450 km) matches Question 5's independently-computed")
print("Dynamic Programming optimum for the same city network ✔ — strong cross-validation evidence")

VERIFIED: TSP brute-force result (450 km) matches Question 5's independently-computed
Dynamic Programming optimum for the same city network ✔ — strong cross-validation evidence


## (c) Exact vs Approximation Methods

| Method | Accuracy | Speed |
|---|---|---|
| Exact Algorithms (DP, Brute Force, Branch & Bound) | Optimal solution | Slower, often exponential/factorial |
| Approximation Algorithms (Greedy, Heuristics) | Near-optimal | Much faster, polynomial time |

**Examples:**
- Knapsack via Dynamic Programming → **Exact** method (pseudo-polynomial O(nW))
- Greedy nearest-neighbour route selection (Question 5) → **Approximation** method

In [5]:
import time

# Empirically show TSP brute force's factorial blow-up as city count grows
import math

def time_tsp(n_cities):
    import random
    random.seed(0)
    dm = [[0 if i==j else random.randint(10,200) for j in range(n_cities)] for i in range(n_cities)]
    start = time.perf_counter()
    best = float('inf')
    for path in permutations(range(1, n_cities)):
        full = (0,) + path
        cost = sum(dm[full[i]][full[i+1]] for i in range(len(full)-1)) + dm[full[-1]][full[0]]
        best = min(best, cost)
    return time.perf_counter() - start

print(f"{'Cities':>8} | {'Permutations':>14} | {'Time (s)':>10}")
for n in [4, 6, 8, 9]:
    perms = math.factorial(n - 1)
    t = time_tsp(n)
    print(f"{n:>8} | {perms:>14,} | {t:>10.5f}")
print("\nThe factorial growth (n-1)! is visible directly in both the permutation count and runtime.")
print("By ~20 cities, brute force TSP becomes computationally infeasible (20! ≈ 2.4 x 10^18).")

  Cities |   Permutations |   Time (s)
       4 |              6 |    0.00002
       6 |            120 |    0.00013
       8 |          5,040 |    0.00592
       9 |         40,320 |    0.05035

The factorial growth (n-1)! is visible directly in both the permutation count and runtime.
By ~20 cities, brute force TSP becomes computationally infeasible (20! ≈ 2.4 x 10^18).


## (d) Computational Complexity Analysis

| Problem | Algorithm | Complexity |
|---|---|---|
| Knapsack (0/1) | Dynamic Programming | O(n·W) — pseudo-polynomial |
| Traveling Salesman | Brute Force | O(n!) |
| Traveling Salesman | Held-Karp DP (see Q5) | O(n² · 2ⁿ) — better than brute force, still exponential |

### Discussion

The Knapsack problem is solvable efficiently with dynamic programming because it has **overlapping subproblems** and **optimal substructure** — the DP table reuses solutions to smaller capacity/item subproblems rather than recomputing them. Note its complexity, O(n·W), is *pseudo-polynomial*: it is polynomial in the numeric value of W, but exponential in the number of bits needed to represent W, so it still degrades for very large capacities.

The Traveling Salesman Problem has no known polynomial-time exact algorithm (it is NP-hard); brute force is O(n!) and the smarter Held-Karp dynamic programming approach (used in Question 5) improves this to O(n² · 2ⁿ) — still exponential, but a substantial improvement that makes solving instances with ~20 cities feasible where brute force is hopeless. Beyond that scale, **approximation algorithms** (e.g. nearest-neighbour, genetic algorithms, simulated annealing) become the only practical option, trading a small amount of accuracy for tractable runtime — exactly the trade-off explored across Question 5's three strategies.